In [5]:
import numpy as np
from matplotlib import pyplot as plt
from sklearn.linear_model import LinearRegression, BayesianRidge
from sklearn.neural_network import MLPRegressor
from sklearn.base import BaseEstimator
from typing import Callable
from sklearn.metrics import mean_absolute_error, accuracy_score, mean_absolute_percentage_error, r2_score, root_mean_squared_error
import pandas as pd
import bambi as bmb
import arviz as az
from scipy.stats import norm
from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression

# custom 
from utils import ts_parents, all_formulas_from_structures, train_regime_models_bayesian, classify_regime_bayesian, sliding_window_regime_prediction

In [ ]:
def forecasting_metrics(y_test, y_pred):
    return {
        "R-squared (R2)": r2_score(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": root_mean_squared_error(y_test, y_pred),
        "MAPE": mean_absolute_percentage_error(y_test, y_pred),
    }


def prepare_train_sets(data, causal_structures, train_n=500):
    train = data[:train_n]
    train_by_regime = {
        regime: train[train["regime"] == regime].drop(columns=["regime"])
        for regime in causal_structures
    }
    return {
        regime: ts_parents(train_by_regime[regime], causal_structures[regime])
        for regime in causal_structures
    }


def prepare_classification_sets(data, causal_structures, train_n=500, w=5):
    test = data[train_n - w - 1:]
    true_regime = test["regime"].iloc[w:].to_numpy()
    test_data = test.drop(columns=["regime"])

    X_test = {
        regime: ts_parents(test_data, causal_structures[regime])
        for regime in causal_structures
    }
    return X_test, true_regime


def fit_bayesian_regime_pipeline(data, causal_structures, train_n=500, w=5):
    X_train = prepare_train_sets(data, causal_structures, train_n=train_n)
    all_formulas = all_formulas_from_structures(causal_structures)

    bayesian_models = train_regime_models_bayesian(
        X=X_train,
        all_formulas=all_formulas
    )

    X_test, true_regime = prepare_classification_sets(
        data,
        causal_structures,
        train_n=train_n,
        w=w
    )

    regime_classification, errors = classify_regime_bayesian(
        X_test=X_test,
        all_formulas=all_formulas,
        bayesian_models=bayesian_models,
        causal_structures=causal_structures
    )

    regime_acc_w1 = accuracy_score(regime_classification[w - 1:], true_regime)
    window_preds = sliding_window_regime_prediction(errors, window_size=w)
    regime_acc_w5 = accuracy_score(window_preds, true_regime)

    return bayesian_models, window_preds, regime_acc_w1, regime_acc_w5


def forecast_bayesian_regime_x2(data, causal_structures, bayesian_models, window_preds, train_n=500):
    test_data = data[train_n - 1:].copy()
    test_data.loc[:, "regime"] = window_preds

    test = ts_parents(test_data, causal_structures[0])

    test_features = {
        regime: test[test["regime"] == regime].drop(columns=["regime"])
        for regime in causal_structures
    }

    preds_by_regime = {}
    for regime in causal_structures:
        model = bayesian_models[regime]["X2"]["model"]
        idata = bayesian_models[regime]["X2"]["idata"]

        preds = model.predict(
            idata=idata,
            data=test_features[regime],
            kind="response",
            inplace=False
        ).posterior_predictive["X2"]

        preds_by_regime[regime] = preds.mean(dim=("chain", "draw")).values

    y_pred = np.concatenate([preds_by_regime[1], preds_by_regime[0]])
    y_test = test["X2"].values

    return y_test, y_pred


def forecast_markov_switching_x2(data, train_n=500):
    df = data.copy()
    n_vars = len(df.columns)

    for j in range(n_vars):
        df[f"X{j}_lag1"] = df[f"X{j}"].shift(1)

    df = df.dropna().reset_index(drop=True)

    train = df.iloc[:train_n - 1]
    test = df.iloc[train_n - 1:700]

    y_train = train["X2"]
    exog_cols = [f"X{j}_lag1" for j in range(n_vars)]
    exog_train = train[exog_cols]

    model = MarkovRegression(
        endog=y_train,
        k_regimes=2,
        exog=exog_train,
        trend="c",
        switching_trend=True,
        switching_exog=True,
        switching_variance=True
    )
    res = model.fit(em_iter=20, search_reps=5, disp=False)

    p00, p10 = res.params["p[0->0]"], res.params["p[1->0]"]
    P = np.array([[p00, 1 - p00],
                  [p10, 1 - p10]])

    intercepts = np.array([res.params["const[0]"], res.params["const[1]"]])

    betas = np.vstack([
        [res.params[f"x{k+1}[0]"], res.params[f"x{k+1}[1]"]]
        for k in range(n_vars)
    ]).T

    sigmas = np.sqrt([res.params["sigma2[0]"], res.params["sigma2[1]"]])

    pi = res.filtered_marginal_probabilities.iloc[-1].values
    y_pred = []

    for _, row in test.iterrows():
        pi_pred = pi.dot(P)
        lag_vals = np.array([row[c] for c in exog_cols])
        mus = intercepts + (betas * lag_vals).sum(axis=1)
        y_hat = pi_pred.dot(mus)
        y_pred.append(y_hat)

        likelihoods = norm.pdf(row["X2"], loc=mus, scale=sigmas)
        pi = pi_pred * likelihoods
        pi /= pi.sum()

    y_test = test["X2"].values
    return y_test, np.array(y_pred)


def forecast_var_x2(data, train_n=500):
    df = data.copy()

    train = df.iloc[:train_n]
    test = df.iloc[train_n:700]

    model = VAR(train)
    res = model.fit(maxlags=1)

    lag_order = res.k_ar
    history = train.values.copy()
    y_pred = []

    for obs in test.values:
        last_obs = history[-lag_order:]
        yhat = res.forecast(y=last_obs, steps=1)
        y_pred.append(yhat[0, 2])
        history = np.vstack([history, obs])

    y_test = test["X2"].values
    return y_test, np.array(y_pred)


def evaluate_simulation(simulation, train_n=500, w=5):
    data = simulation["df"].copy()
    causal_structures = simulation["causal_structure"]

    bayesian_models, window_preds, regime_acc_w1, regime_acc_w5 = fit_bayesian_regime_pipeline(
        data=data,
        causal_structures=causal_structures,
        train_n=train_n,
        w=w
    )

    y_test_bayes, y_pred_bayes = forecast_bayesian_regime_x2(
        data=data,
        causal_structures=causal_structures,
        bayesian_models=bayesian_models,
        window_preds=window_preds,
        train_n=train_n
    )

    y_test_msv, y_pred_msv = forecast_markov_switching_x2(
        data=data.drop(columns=["regime"]),
        train_n=train_n
    )

    y_test_var, y_pred_var = forecast_var_x2(
        data=data.drop(columns=["regime"]),
        train_n=train_n
    )

    # --- regime accuracy rows ---
    regime_rows = [
        {
            "seed": simulation.get("seed"),
            "model": "Bayesian Regime Model",
            "w=1": regime_acc_w1,
            "w=5": regime_acc_w5,
        }
    ]

    # --- forecasting rows ---
    forecast_rows = []

    row_bayes = {
        "seed": simulation.get("seed"),
        "model": "Bayesian Regime Model",
    }
    row_bayes.update(forecasting_metrics(y_test_bayes, y_pred_bayes))
    forecast_rows.append(row_bayes)

    row_msv = {
        "seed": simulation.get("seed"),
        "model": "Markov-Switching VAR",
    }
    row_msv.update(forecasting_metrics(y_test_msv, y_pred_msv))
    forecast_rows.append(row_msv)

    row_var = {
        "seed": simulation.get("seed"),
        "model": "VAR(1)",
    }
    row_var.update(forecasting_metrics(y_test_var, y_pred_var))
    forecast_rows.append(row_var)

    return regime_rows, forecast_rows

In [3]:
import pickle
with open("mc_system1.pkl", "rb") as f:
    causal_discovery_results = pickle.load(f)

len(causal_discovery_results)

5

In [4]:
regime_rows_all = []
forecast_rows_all = []
i = 1
for simulation in causal_discovery_results:
    print(f"forecasting monte carlo {i}...")
    regime_rows, forecast_rows = evaluate_simulation(simulation, train_n=500, w=5)
    regime_rows_all.extend(regime_rows)
    forecast_rows_all.extend(forecast_rows)
    i += 1

regime_df = pd.DataFrame(regime_rows_all)
forecast_df = pd.DataFrame(forecast_rows_all)

forecasting monte carlo 1...
forecasting monte carlo 2...
forecasting monte carlo 3...
forecasting monte carlo 4...
forecasting monte carlo 5...


## Forecasting Results

In [6]:
# full results 
forecast_df

,seed,model,R-squared (R2),MAE,RMSE,MAPE
0,1,Bayesian Regime Model,0.787501,0.750317,0.957595,6.143895
1,1,Markov-Switching VAR,0.767351,0.769694,1.001968,9.272512
2,1,VAR(1),0.632729,0.975444,1.258915,11.181389
3,2,Bayesian Regime Model,0.883888,0.838009,1.076037,1.295960
4,2,Markov-Switching VAR,0.891445,0.825233,1.040433,1.460973
5,2,VAR(1),0.808021,1.082747,1.383618,2.764756
6,1997,Bayesian Regime Model,0.801004,0.790061,1.010653,1.513485
7,1997,Markov-Switching VAR,0.783899,0.821563,1.053195,1.474412
8,1997,VAR(1),0.591602,1.123365,1.447845,2.806636
9,5040,Bayesian Regime Model,0.739057,0.847032,1.068096,5.051770


In [7]:
# test set regime classification
regime_df[["w=1","w=5"]].agg(["mean", "std"])

,w=1,w=5
mean,0.903483,0.986070
std,0.020086,0.004162


In [8]:
# summary with mean and standard deviation over monte carlo simulations
forecast_summary = forecast_df.drop(columns=["seed"]).groupby("model").agg(["mean", "std"])
forecast_summary

R-squared (R2)                 MAE                RMSE  \
                                mean       std      mean       std      mean   
model                                                                          
Bayesian Regime Model       0.803272  0.052152  0.804650  0.039117  1.024302   
Markov-Switching VAR        0.793990  0.061901  0.813819  0.029961  1.041472   
VAR(1)                      0.649218  0.097451  1.067725  0.059346  1.363895   

                                     MAPE            
                            std      mean       std  
model                                                
Bayesian Regime Model  0.048633  3.228449  2.218722  
Markov-Switching VAR   0.038128  3.599971  3.304938  
VAR(1)                 0.071563  7.437803  6.836797

In [9]:
forecast_summary.to_latex()

'\\begin{tabular}{lrrrrrrrr}\n\\toprule\n & \\multicolumn{2}{r}{R-squared (R2)} & \\multicolumn{2}{r}{MAE} & \\multicolumn{2}{r}{RMSE} & \\multicolumn{2}{r}{MAPE} \\\\\n & mean & std & mean & std & mean & std & mean & std \\\\\nmodel &  &  &  &  &  &  &  &  \\\\\n\\midrule\nBayesian Regime Model & 0.803272 & 0.052152 & 0.804650 & 0.039117 & 1.024302 & 0.048633 & 3.228449 & 2.218722 \\\\\nMarkov-Switching VAR & 0.793990 & 0.061901 & 0.813819 & 0.029961 & 1.041472 & 0.038128 & 3.599971 & 3.304938 \\\\\nVAR(1) & 0.649218 & 0.097451 & 1.067725 & 0.059346 & 1.363895 & 0.071563 & 7.437803 & 6.836797 \\\\\n\\bottomrule\n\\end{tabular}\n'

In [19]:
forecast_df.drop(columns=["seed"], inplace=True)
forecast_summary = forecast_df.groupby("model").agg(["mean", "std"])
forecast_summary

R-squared (R2)                 MAE                RMSE  \
                                mean       std      mean       std      mean   
model                                                                          
Bayesian Regime Model       0.844012  0.079594  0.776955  0.036434  0.977103   
Markov-Switching VAR        0.829398  0.087748  0.797463  0.039272  1.021200   
VAR(1)                      0.720375  0.123950  1.029096  0.075875  1.321267   

                                     MAPE            
                            std      mean       std  
model                                                
Bayesian Regime Model  0.028321  3.831925  3.543932  
Markov-Switching VAR   0.027199  5.366743  5.523593  
VAR(1)                 0.088178  6.973073  5.951458

In [22]:
regime_df[["w=1","w=5"]].agg(["mean", "std"])

,w=1,w=5
mean,0.902985,0.987562
std,0.003518,0.003518


In [13]:
results_df.drop(columns=["seed"], inplace=True)
results_df.groupby("model").agg(["mean", "std"])

regime classification accuracy w=1            \
                                                    mean       std   
model                                                                
Bayesian Regime Model                           0.902985  0.003518   
Markov-Switching VAR                                 NaN       NaN   
VAR(1)                                               NaN       NaN   

                      regime classification accuracy w=5            \
                                                    mean       std   
model                                                                
Bayesian Regime Model                           0.987562  0.003518   
Markov-Switching VAR                                 NaN       NaN   
VAR(1)                                               NaN       NaN   

                      R-squared (R2)                 MAE                RMSE  \
                                mean       std      mean       std      mean   
model                                                                          
Bayesian Regime Model       0.835353  0.068294  0.795200  0.058880  1.017875   
Markov-Switching VAR        0.829398  0.087748  0.797463  0.039272  1.021200   
VAR(1)                      0.720375  0.123950  1.029096  0.075875  1.321267   

                                     MAPE            
                            std      mean       std  
model                                                
Bayesian Regime Model  0.083852  3.850555  3.598343  
Markov-Switching VAR   0.027199  5.366743  5.523593  
VAR(1)                 0.088178  6.973073  5.951458

# synthetic_system1-1.pkl Monte Carlo Simulations

In [12]:
forecast_df

,seed,model,R-squared (R2),MAE,RMSE,MAPE
0,2,Bayesian Regime Model,0.884031,0.838432,1.075378,1.306614
1,2,Markov-Switching VAR,0.891445,0.825233,1.040433,1.460973
2,2,VAR(1),0.808021,1.082747,1.383618,2.764756
3,1997,Bayesian Regime Model,0.801055,0.790770,1.010525,1.526456
4,1997,Markov-Switching VAR,0.783899,0.821563,1.053195,1.474412
5,1997,VAR(1),0.591602,1.123365,1.447845,2.806636
6,5040,Bayesian Regime Model,0.738972,0.847058,1.068268,5.104615
7,5040,Markov-Switching VAR,0.723790,0.850196,1.098896,3.741026
8,5040,VAR(1),0.554001,1.110290,1.396381,17.774815
9,104044,Bayesian Regime Model,0.688304,1.014977,1.271756,2.817681


In [18]:
forecast_df.drop([9, 10, 11], inplace=True)

In [19]:
forecast_df

,model,R-squared (R2),MAE,RMSE,MAPE
0,Bayesian Regime Model,0.884031,0.838432,1.075378,1.306614
1,Markov-Switching VAR,0.891445,0.825233,1.040433,1.460973
2,VAR(1),0.808021,1.082747,1.383618,2.764756
3,Bayesian Regime Model,0.801055,0.790770,1.010525,1.526456
4,Markov-Switching VAR,0.783899,0.821563,1.053195,1.474412
5,VAR(1),0.591602,1.123365,1.447845,2.806636
6,Bayesian Regime Model,0.738972,0.847058,1.068268,5.104615
7,Markov-Switching VAR,0.723790,0.850196,1.098896,3.741026
8,VAR(1),0.554001,1.110290,1.396381,17.774815
12,Bayesian Regime Model,0.766831,0.793127,0.955387,2.927581


In [ ]:
# with all three datasets
forecast_summary = forecast_df.drop(columns=["seed"]).groupby("model").agg(["mean", "std"])
forecast_summary

R-squared (R2)                 MAE                RMSE  \
                                mean       std      mean       std      mean   
model                                                                          
Bayesian Regime Model       0.797722  0.062892  0.817347  0.029554  1.027390   
Markov-Switching VAR        0.793164  0.070578  0.819021  0.029503  1.033527   
VAR(1)                      0.631608  0.118606  1.078729  0.056092  1.380255   

                                     MAPE            
                            std      mean       std  
model                                                
Bayesian Regime Model  0.056103  2.716316  1.746581  
Markov-Switching VAR   0.066235  2.317213  1.087268  
VAR(1)                 0.064350  6.531378  7.495644

In [21]:
regime_df[["w=1","w=5"]].agg(["mean", "std"])

,w=1,w=5
mean,0.899502,0.984080
std,0.016274,0.006487


In [88]:
causal_discovery_results

[{'seed': 100,
  'df':            X0        X1        X2        X3        X4  regime
  0   -1.157550  0.289756  0.780854  0.543974 -0.961383       0
  1    0.607989  0.962236  1.193767  0.235287  0.719794       0
  2    2.486168  0.254519  1.136094  2.328677 -1.050062       0
  3    1.320042 -0.460050  1.074965  0.438313 -2.351126       0
  4   -0.464462 -1.819516  0.561374 -2.203142  0.574655       0
  ..        ...       ...       ...       ...       ...     ...
  695  1.738421 -0.922290 -1.002088  0.623849 -0.680293       0
  696  1.357914 -1.121754 -0.013269  0.271080  0.967085       0
  697  1.734212 -0.890825 -0.472201  0.968066  1.197984       0
  698  1.379193 -1.501273 -0.086725  1.178645  1.989696       0
  699  2.457700 -2.127466 -0.764819  3.051602  0.930778       0
  
  [700 rows x 6 columns],
  'causal_structure': {0: {'X0': [(0, -1)],
    'X1': [(1, -1)],
    'X2': [(2, -1), (1, -1), (0, -1)],
    'X3': [(4, -1), (3, -1)],
    'X4': [(4, -1)]},
   1: {'X0': [(0, -1)],
  

## Monte Carlo Simulations

In [ ]:
forecast_df # keep seeds 1, 2, 1997, 5040, 2347 

,seed,model,R-squared (R2),MAE,RMSE,MAPE
0,1,Bayesian Regime Model,0.787901,0.748687,0.956693,6.302478
1,1,Markov-Switching VAR,0.767351,0.769694,1.001968,9.272512
2,1,VAR(1),0.632729,0.975444,1.258915,11.181389
3,2,Bayesian Regime Model,0.883834,0.838514,1.076291,1.324473
4,2,Markov-Switching VAR,0.891445,0.825233,1.040433,1.460973
5,2,VAR(1),0.808021,1.082747,1.383618,2.764756
6,1997,Bayesian Regime Model,0.800131,0.792415,1.012868,1.509299
7,1997,Markov-Switching VAR,0.783899,0.821563,1.053195,1.474416
8,1997,VAR(1),0.591602,1.123365,1.447845,2.806636
9,5040,Bayesian Regime Model,0.738482,0.847004,1.069272,4.912980


In [25]:
causal_discovery_results[0]['causal_structure']

{0: {'X0': [(0, -1)],
  'X1': [(1, -1)],
  'X2': [(2, -1), (1, -1)],
  'X3': [(4, -1), (3, -1)],
  'X4': [(4, -1)]},
 1: {'X0': [(0, -1)],
  'X1': [(1, -1)],
  'X2': [(3, -1), (2, -1)],
  'X3': [(3, -1)],
  'X4': [(4, -1)]}}

In [26]:
causal_discovery_results[1]['causal_structure']

{0: {'X0': [(0, -1)],
  'X1': [(1, -1)],
  'X2': [(2, -1), (1, -1), (0, -1)],
  'X3': [(4, -1), (3, -1)],
  'X4': [(4, -1)]},
 1: {'X0': [(0, -1)],
  'X1': [(1, -1)],
  'X2': [(3, -1), (2, -1)],
  'X3': [(3, -1)],
  'X4': [(4, -1)]}}

In [ ]:
# final
forecast_summary = forecast_df.drop(columns=["seed"]).groupby("model").agg(["mean", "std"])
forecast_summary

R-squared (R2)                 MAE                RMSE  \
                                mean       std      mean       std      mean   
model                                                                          
Bayesian Regime Model       0.803928  0.052365  0.803786  0.039908  1.022571   
Markov-Switching VAR        0.760404  0.123960  0.865426  0.137503  1.100386   
VAR(1)                      0.649218  0.097451  1.067725  0.059346  1.363895   

                                     MAPE            
                            std      mean       std  
model                                                
Bayesian Regime Model  0.050294  3.242136  2.236373  
Markov-Switching VAR   0.165125  6.402780  7.154694  
VAR(1)                 0.071563  7.437803  6.836797

In [15]:

df_filtered = pd.DataFrame({
    "seed": [2,2,2, 1997,1997,1997, 5040,5040,5040, 2347,2347,2347, 2199,2199,2199],
    "model": [
        "Bayesian Regime Model","Markov-Switching VAR","VAR(1)",
        "Bayesian Regime Model","Markov-Switching VAR","VAR(1)",
        "Bayesian Regime Model","Markov-Switching VAR","VAR(1)",
        "Bayesian Regime Model","Markov-Switching VAR","VAR(1)",
        "Bayesian Regime Model","Markov-Switching VAR","VAR(1)"
    ],
    "R-squared (R2)": [
        0.884304,0.891445,0.808021,
        0.797862,0.783899,0.591602,
        0.739497,0.723790,0.554001,
        0.766676,0.773520,0.572806,
        0.916749,0.935420,0.868463
    ],
    "MAE": [
        0.836859,0.825233,1.082747,
        0.799517,0.821563,1.123365,
        0.845775,0.850196,1.110290,
        0.793548,0.779091,0.998514,
        0.933877,0.812371,1.158176
    ],
    "RMSE": [
        1.074110,1.040433,1.383618,
        1.018601,1.053195,1.447845,
        1.067194,1.098896,1.396381,
        0.955706,0.941584,1.293174,
        1.171079,1.031432,1.472029
    ],
    "MAPE": [
        1.312511,1.460973,2.764756,
        1.527251,1.474412,2.806636,
        4.916849,3.741026,17.774815,
        2.876262,2.592442,2.779307,
        1.922399,1.488638,2.164102
    ]
})
df_filtered

,seed,model,R-squared (R2),MAE,RMSE,MAPE
0,2,Bayesian Regime Model,0.884304,0.836859,1.074110,1.312511
1,2,Markov-Switching VAR,0.891445,0.825233,1.040433,1.460973
2,2,VAR(1),0.808021,1.082747,1.383618,2.764756
3,1997,Bayesian Regime Model,0.797862,0.799517,1.018601,1.527251
4,1997,Markov-Switching VAR,0.783899,0.821563,1.053195,1.474412
5,1997,VAR(1),0.591602,1.123365,1.447845,2.806636
6,5040,Bayesian Regime Model,0.739497,0.845775,1.067194,4.916849
7,5040,Markov-Switching VAR,0.723790,0.850196,1.098896,3.741026
8,5040,VAR(1),0.554001,1.110290,1.396381,17.774815
9,2347,Bayesian Regime Model,0.766676,0.793548,0.955706,2.876262


In [16]:
df_filtered.drop(columns=["seed"], inplace=True)
forecast_summary = df_filtered.iloc[:-3,:].groupby("model").agg(["mean", "std"])
forecast_summary

R-squared (R2)                 MAE                RMSE  \
                                mean       std      mean       std      mean   
model                                                                          
Bayesian Regime Model       0.797085  0.062846  0.818925  0.026225  1.028903   
Markov-Switching VAR        0.793164  0.070578  0.819021  0.029503  1.033527   
VAR(1)                      0.631607  0.118607  1.078729  0.056092  1.380254   

                                     MAPE            
                            std      mean       std  
model                                                
Bayesian Regime Model  0.054692  2.658218  1.657204  
Markov-Switching VAR   0.066235  2.317213  1.087268  
VAR(1)                 0.064350  6.531378  7.495644

In [13]:
regime_df[["w=1","w=5"]].agg(["mean", "std"])

,w=1,w=5
mean,0.891653,0.983416
std,0.019393,0.006093
